In [ ]:
import pandas as pd

# 1) Cargar CSV (ajusta la ruta si lo guardaste en otra parte)
csv_path = "movimientos_cartola_2025-12_a_2026-01.csv"
df = pd.read_csv(csv_path)

# 2) Parsear fecha y asegurar tipos
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["monto_clp"] = pd.to_numeric(df["monto_clp"], errors="coerce")

# 3) Quick check
print("Filas:", len(df), "| Columnas:", list(df.columns))
df.head()


In [ ]:
import numpy as np

# Normalización mínima: solo UBER
df["descripcion_norm"] = df["descripcion"].replace({
    "PAGO UBER TRIP": "UBER",
    "PAGO UBER": "UBER",
})

# (Opcional) Si quieres que funcione aunque venga con variaciones:
# df["descripcion_norm"] = np.where(df["descripcion"].str.contains(r"\bUBER\b", na=False), "UBER", df["descripcion"])

df[["descripcion", "descripcion_norm"]].drop_duplicates().sort_values("descripcion").head(30)


In [ ]:
gastos_por_desc = (
    df[df["monto_clp"] < 0]
      .groupby("descripcion_norm", as_index=False)["monto_clp"]
      .sum()
)

gastos_por_desc["monto_gastado_clp"] = -gastos_por_desc["monto_clp"]

top_gastos = (gastos_por_desc
              .sort_values("monto_gastado_clp", ascending=False)
              .head(20)
              .reset_index(drop=True)
              .drop(columns=["monto_clp"]))

top_gastos


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# Assume df and top_gastos exists; if not, compute quickly
if 'df' not in globals():
    df = pd.read_csv("/mnt/data/movimientos_cartola_2025-12_a_2026-01.csv")
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["monto_clp"] = pd.to_numeric(df["monto_clp"], errors="coerce")

gastos_por_desc = (
    df[df["monto_clp"] < 0]
      .groupby("descripcion", as_index=False)["monto_clp"]
      .sum()
)
gastos_por_desc["monto_gastado_clp"] = -gastos_por_desc["monto_clp"]
TOP_N = 20
top_gastos = (
    gastos_por_desc
      .sort_values("monto_gastado_clp", ascending=False)
      .head(TOP_N)
      .reset_index(drop=True)
      .drop(columns=["monto_clp"])
)

# Plot 1: Top descriptions horizontal bar
plt.figure(figsize=(10, 7))
plt.barh(top_gastos["descripcion"][::-1], top_gastos["monto_gastado_clp"][::-1])
plt.xlabel("Gasto total (CLP)")
plt.ylabel("Descripción")
plt.title(f"Top {TOP_N} gastos por descripción")
plt.tight_layout()
plt.show()

# Plot 2: Daily spend trend
daily_spend = (
    df[df["monto_clp"] < 0]
      .groupby("fecha", as_index=False)["monto_clp"]
      .sum()
)
daily_spend["gasto_diario_clp"] = -daily_spend["monto_clp"]
daily_spend = daily_spend.sort_values("fecha")

plt.figure(figsize=(10, 4))
plt.plot(daily_spend["fecha"], daily_spend["gasto_diario_clp"])
plt.xlabel("Fecha")
plt.ylabel("Gasto diario (CLP)")
plt.title("Tendencia de gasto diario")
plt.tight_layout()
plt.show()

daily_spend.tail(), top_gastos.head()



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_N = 12  # baja esto para menos etiquetas y gráfico más legible (10-15 suele andar bien)

# Agrupar gastos por descripcion_norm
gastos = (
    df[df["monto_clp"] < 0]
      .groupby("descripcion_norm", as_index=False)["monto_clp"]
      .sum()
)
gastos["monto_gastado_clp"] = -gastos["monto_clp"]

top = gastos.sort_values("monto_gastado_clp", ascending=False).reset_index(drop=True)

topN = top.head(TOP_N).copy()
otros_total = top.iloc[TOP_N:]["monto_gastado_clp"].sum()

if otros_total > 0:
    topN = pd.concat([topN, pd.DataFrame([{
        "descripcion_norm": "OTROS",
        "monto_gastado_clp": otros_total
    }])], ignore_index=True)

total_gasto = top["monto_gastado_clp"].sum()

labels = topN["descripcion_norm"].tolist()
values = topN["monto_gastado_clp"].tolist()

def make_autopct(vals):
    total = sum(vals)
    def _autopct(pct):
        val = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n${val:,.0f}"
    return _autopct

plt.figure(figsize=(14, 10))  # MÁS GRANDE
plt.pie(
    values,
    labels=labels,
    autopct=make_autopct(values),  # % + $valor
    startangle=90,
    textprops={"fontsize": 11},
    pctdistance=0.72,
    labeldistance=1.08
)
plt.title(f"Distribución de gastos (Total: ${total_gasto:,.0f} CLP)", fontsize=16)
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

tmp = df[df["monto_clp"] < 0].copy()
tmp["dow"] = tmp["fecha"].dt.day_name()  # en inglés por default

gasto_dow = (
    tmp.groupby("dow", as_index=False)["monto_clp"].sum()
)
gasto_dow["gasto_clp"] = -gasto_dow["monto_clp"]

# Orden típico semana (ajusta si quieres lunes primero)
orden = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
gasto_dow["dow"] = pd.Categorical(gasto_dow["dow"], categories=orden, ordered=True)
gasto_dow = gasto_dow.sort_values("dow")

plt.figure(figsize=(8, 4))
plt.bar(gasto_dow["dow"].astype(str), gasto_dow["gasto_clp"])
plt.xlabel("Día de semana")
plt.ylabel("Gasto total (CLP)")
plt.title("Gasto total por día de semana")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure df exists
if 'df' not in globals():
    df = pd.read_csv("/mnt/data/movimientos_cartola_2025-12_a_2026-01.csv")
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["monto_clp"] = pd.to_numeric(df["monto_clp"], errors="coerce")

# Build daily totals (positive spend)
daily = df[df["monto_clp"] < 0].groupby("fecha")["monto_clp"].sum().rename("monto_clp").reset_index()
daily["gasto_diario_clp"] = -daily["monto_clp"]
daily["dow"] = daily["fecha"].dt.dayofweek  # Monday=0
daily["week_start"] = daily["fecha"] - pd.to_timedelta(daily["dow"], unit="D")

# Pivot week vs day-of-week
pivot = daily.pivot_table(index="week_start", columns="dow", values="gasto_diario_clp", aggfunc="sum", fill_value=0)

# Ensure all 0..6 columns exist
for c in range(7):
    if c not in pivot.columns:
        pivot[c] = 0
pivot = pivot[[0,1,2,3,4,5,6]]

# Plot heatmap (matplotlib imshow)
plt.figure(figsize=(10, 5))
plt.imshow(pivot.values, aspect="auto")
plt.yticks(range(len(pivot.index)), [d.strftime("%Y-%m-%d") for d in pivot.index])
plt.xticks(range(7), ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"], rotation=0)
plt.xlabel("Día de semana")
plt.ylabel("Semana (inicio)")
plt.title("Mapa de calor: gasto diario (CLP) por día de semana")

cbar = plt.colorbar()
cbar.set_label("Gasto diario (CLP)")

plt.tight_layout()
plt.show()

pivot.head()



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Mapa de calor: gastos del mes completo (día del mes x día de semana) ===
# Asume df ya cargado y df["fecha"] es datetime, df["monto_clp"] numérico

# 1) Filtrar solo gastos (cargos) y convertir a positivo
g = df[df["monto_clp"] < 0].copy()
g["gasto_clp"] = -g["monto_clp"]

# 2) Elegir el mes a graficar (por defecto: el último mes presente en el DF)
mes = g["fecha"].dt.to_period("M").max()
gm = g[g["fecha"].dt.to_period("M") == mes].copy()

# 3) Preparar ejes: día del mes (1..31) vs día semana (Lun..Dom)
gm["dia_mes"] = gm["fecha"].dt.day
gm["dow"] = gm["fecha"].dt.dayofweek  # 0=Lun ... 6=Dom

heat = (gm.groupby(["dia_mes", "dow"])["gasto_clp"].sum().unstack(fill_value=0))

# asegurar columnas 0..6 y filas 1..31 (aunque el mes tenga menos días)
heat = heat.reindex(columns=range(7), fill_value=0).reindex(index=range(1, 32), fill_value=np.nan)

# 4) Plot (matplotlib)
plt.figure(figsize=(11, 6))
plt.imshow(heat.values, aspect="auto")

plt.xticks(range(7), ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"])
plt.yticks(range(31), [str(i) for i in range(1, 32)])

plt.xlabel("Día de semana")
plt.ylabel("Día del mes")
plt.title(f"Mapa de calor de gastos (CLP) - {mes}")

cbar = plt.colorbar()
cbar.set_label("Gasto (CLP)")

plt.tight_layout()
plt.show()


In [ ]:
# Top descripciones por cantidad de movimientos
conteo_desc = df["descripcion"].value_counts().reset_index()
conteo_desc.columns = ["descripcion", "n_movimientos"]
conteo_desc.head(30)


In [ ]:
repetidas = (
    df["descripcion"]
      .value_counts()
      .reset_index()
)

repetidas.columns = ["descripcion", "n"]
repetidas = repetidas[repetidas["n"] >= 2].sort_values("n", ascending=False)

repetidas.head(50)
